In [10]:
# Customer Support Ticket Auto-Triage System

In [11]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import time
import pickle
import json

In [12]:
np.random.seed(42)

n_samples = 5000
categories = ['Bug Report', 'Feature Request', 'Technical Issue', 'Billing Inquiry', 'Account Management']
priorities = ['Low', 'Medium', 'High', 'Critical']

# Sample templates for each category
templates = {
    'Bug Report': [
        ('App crashes when uploading', 'The application crashes every time I try to upload a file larger than 10MB'),
        ('Error 404 on dashboard', 'Getting error 404 when accessing the main dashboard page'),
        ('Login button not working', 'Cannot click the login button on mobile devices')
    ],
    'Feature Request': [
        ('Add dark mode', 'Would love to have a dark mode option for night time usage'),
        ('Export to PDF needed', 'Please add functionality to export reports as PDF'),
        ('Bulk upload feature', 'Need ability to upload multiple files at once')
    ],
    'Technical Issue': [
        ('API connection timeout', 'Getting timeout errors when connecting to the API endpoint'),
        ('Database sync issues', 'Data not syncing properly between mobile and web versions'),
        ('Performance degradation', 'System becomes very slow after 2 hours of usage')
    ],
    'Billing Inquiry': [
        ('Wrong charge amount', 'I was charged $99 instead of the advertised $79'),
        ('Refund request', 'Need a refund for accidental duplicate subscription'),
        ('Payment method update', 'Unable to update my credit card information')
    ],
    'Account Management': [
        ('Password reset not working', 'Not receiving password reset emails'),
        ('Delete account request', 'Please help me delete my account permanently'),
        ('Change email address', 'Need to update my registered email address')
    ]
}

data = []
for i in range(n_samples):
    category = np.random.choice(categories)
    subject, description = templates[category][np.random.randint(0, 3)]
    
    # Add variations
    subject += f" #{i%100}"
    description += f" Issue ID: {i}. This has been happening for {np.random.randint(1,30)} days."
    
    data.append({
        'Ticket_ID': i + 1000,
        'Subject': subject,
        'Description': description,
        'Category': category,
        'Priority': np.random.choice(priorities),
        'Timestamp': pd.Timestamp.now() - pd.Timedelta(days=np.random.randint(0, 365))
    })

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"\nCategory distribution:\n{df['Category'].value_counts()}")

Dataset shape: (5000, 6)

Category distribution:
Category
Feature Request       1051
Account Management    1004
Technical Issue        989
Billing Inquiry        978
Bug Report             978
Name: count, dtype: int64


In [13]:
df['text'] = df['Subject'] + ' ' + df['Description']

label_encoder = LabelEncoder()
df['category_encoded'] = label_encoder.fit_transform(df['Category'])

X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values, 
    df['category_encoded'].values,
    test_size=0.2, 
    random_state=42,
    stratify=df['category_encoded']
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 4000
Test samples: 1000


In [14]:
max_features = 5000
sequence_length = 200

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)

vectorize_layer.adapt(X_train)

vocab = vectorize_layer.get_vocabulary()
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 4146


In [15]:
def create_model():
    model = tf.keras.Sequential([
        # Input layer
        tf.keras.layers.Input(shape=(1,), dtype=tf.string),
        
        # Text vectorization
        vectorize_layer,
        
        # Embedding layer
        tf.keras.layers.Embedding(max_features, 128),
        
        # LSTM for sequence processing
        tf.keras.layers.LSTM(64, dropout=0.5),
        
        # Dense layers
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        
        # Output layer
        tf.keras.layers.Dense(len(categories), activation='softmax')
    ])
    
    return model

model = create_model()

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 200)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 691,653 (2.64 MB)

 Trainable params: 691,653 (2.64 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train model
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10


100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 95ms/step - accuracy: 0.2078 - loss: 1.6115 - val_accuracy: 0.1838 - val_loss: 1.6117
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 99ms/step - accuracy: 0.1963 - loss: 1.6106 - val_accuracy: 0.2163 - val_loss: 1.6101
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step - accuracy: 0.2025 - loss: 1.6103 - val_accuracy: 0.2163 - val_loss: 1.6093
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step - accuracy: 0.2050 - loss: 1.6100 - val_accuracy: 0.2163 - val_loss: 1.6094
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 95ms/step - accuracy: 0.2041 - loss: 1.6095 - val_accuracy: 0.2163 - val_loss: 1.6101
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step - accuracy: 0.2006 - loss: 1.6098 - val_accuracy: 0.2163 - val_loss: 1.6095


In [17]:
# Predictions
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)

# Calculate metrics as per PDF requirements
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

# Measure latency
latency_times = []
for i in range(100):
    start_time = time.time()
    _ = model.predict(X_test[i:i+1], verbose=0)
    latency_times.append(time.time() - start_time)
avg_latency = np.mean(latency_times) * 1000  # in milliseconds

print("="*50)
print("MODEL EVALUATION METRICS")
print("="*50)
print(f"Accuracy (40% weight): {accuracy:.4f}")
print(f"Precision (15% weight): {precision:.4f}")
print(f"Recall (15% weight): {recall:.4f}")
print(f"F1-Score (20% weight): {f1:.4f}")
print(f"Avg Latency (10% weight): {avg_latency:.2f} ms")
print("\nWeighted Performance Score:")
weighted_score = (accuracy * 0.4) + (precision * 0.15) + (recall * 0.15) + (f1 * 0.2) + ((1 - min(avg_latency/100, 1)) * 0.1)
print(f"Overall Score: {weighted_score:.4f}")

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step


/home/deb/anaconda3/envs/phis/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


MODEL EVALUATION METRICS
Accuracy (40% weight): 0.2100
Precision (15% weight): 0.0441
Recall (15% weight): 0.2100
F1-Score (20% weight): 0.0729
Avg Latency (10% weight): 87.82 ms

Weighted Performance Score:
Overall Score: 0.1489


In [18]:
# Detailed classification report
print("\nDETAILED CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred, target_names=categories))


DETAILED CLASSIFICATION REPORT
                    precision    recall  f1-score   support

        Bug Report       0.00      0.00      0.00       201
   Feature Request       0.00      0.00      0.00       196
   Technical Issue       0.00      0.00      0.00       195
   Billing Inquiry       0.21      1.00      0.35       210
Account Management       0.00      0.00      0.00       198

          accuracy                           0.21      1000
         macro avg       0.04      0.20      0.07      1000
      weighted avg       0.04      0.21      0.07      1000



/home/deb/anaconda3/envs/phis/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/deb/anaconda3/envs/phis/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/deb/anaconda3/envs/phis/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is